In [102]:
import pandas as pd
import numpy as np
import pickle
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

from tensorflow.keras.utils import to_categorical

In [103]:
df = pd.read_csv("feedback_dataset.csv")

df["sentiment"] = df["sentiment"].astype(str).str.strip()
df["sentiment"] = df["sentiment"].str.title()

df = df[df["sentiment"] != "Sentiment"]

df = df[df["sentiment"].isin([
    "Positive",
    "Neutral",
    "Negative"
])]

print(df["sentiment"].unique())
print(df.shape)

['Positive' 'Neutral' 'Negative']
(66, 2)


In [104]:
X = df["feedback"]
y = df["sentiment"]

In [105]:
os.makedirs("model", exist_ok=True)

encoder = LabelEncoder()

y = encoder.fit_transform(y)

print(encoder.classes_)

with open("model/label_encoder.pkl", "wb") as f:
    pickle.dump(encoder, f)

['Negative' 'Neutral' 'Positive']


In [106]:
tokenizer = Tokenizer(num_words=5000)

tokenizer.fit_on_texts(X)

X_seq = tokenizer.texts_to_sequences(X)

with open("model/tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [107]:
X_pad = pad_sequences(
    X_seq,
    maxlen=100
)

y = to_categorical(y)

print(y.shape)

(66, 3)


In [108]:
X_train, X_test, y_train, y_test = train_test_split(
    X_pad,
    y,
    test_size=0.2,
    random_state=42
)

In [109]:
model = Sequential([
    Embedding(5000, 128, input_length=100),
    LSTM(128),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dense(3, activation='softmax')
])

c:\Users\sanju\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [110]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [111]:
history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 4s 646ms/step - accuracy: 0.3171 - loss: 1.0980 - val_accuracy: 0.3636 - val_loss: 1.0954
Epoch 2/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 148ms/step - accuracy: 0.4146 - loss: 1.0877 - val_accuracy: 0.3636 - val_loss: 1.0932
Epoch 3/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 161ms/step - accuracy: 0.4146 - loss: 1.0786 - val_accuracy: 0.3636 - val_loss: 1.0926
Epoch 4/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 188ms/step - accuracy: 0.3902 - loss: 1.0768 - val_accuracy: 0.3636 - val_loss: 1.0926
Epoch 5/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 307ms/step - accuracy: 0.3902 - loss: 1.0685 - val_accuracy: 0.3636 - val_loss: 1.0927
Epoch 6/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step - accuracy: 0.4146 - loss: 1.0707 - val_accuracy: 0.3636 - val_loss: 1.0919
Epoch 7/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 175ms/step - accuracy: 0.4390 - loss: 1.0464 - val_accuracy: 0.3636 - val_loss: 1.0919
Epoch 8/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 163ms/step - accuracy: 0.4146 - loss: 1.0457 - val_accuracy: 0.3636 - val_loss:

In [112]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Accuracy:", accuracy)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - accuracy: 0.2857 - loss: 1.1312
Accuracy: 0.2857142984867096


In [113]:
model.save("model/feedback_model.h5")

In [114]:
print(encoder.classes_)
print(y.shape)

['Negative' 'Neutral' 'Positive']
(66, 3)


In [115]:
history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 199ms/step - accuracy: 0.4146 - loss: 1.0023 - val_accuracy: 0.3636 - val_loss: 1.0906
Epoch 2/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step - accuracy: 0.4634 - loss: 1.0140 - val_accuracy: 0.3636 - val_loss: 1.0843
Epoch 3/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step - accuracy: 0.6585 - loss: 0.9615 - val_accuracy: 0.4545 - val_loss: 1.0805
Epoch 4/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 146ms/step - accuracy: 0.6585 - loss: 0.9296 - val_accuracy: 0.4545 - val_loss: 1.0804
Epoch 5/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 159ms/step - accuracy: 0.6585 - loss: 0.9106 - val_accuracy: 0.3636 - val_loss: 1.0826
Epoch 6/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step - accuracy: 0.6098 - loss: 0.8878 - val_accuracy: 0.4545 - val_loss: 1.0854
Epoch 7/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step - accuracy: 0.6829 - loss: 0.8353 - val_accuracy: 0.4545 - val_loss: 1.0888
Epoch 8/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step - accuracy: 0.6829 - loss: 0.7910 - val_accuracy: 0.4545 - val_loss:

In [116]:
model.evaluate(X_test,y_test)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step - accuracy: 0.5000 - loss: 1.0354


[1.0354316234588623, 0.5]

In [117]:
model.save(
    "model/feedback_model.h5"
)

In [118]:
print(df["sentiment"].unique())

['Positive' 'Neutral' 'Negative']


In [119]:
Dense(3, activation='softmax')

<Dense name=dense_15, built=False>

In [120]:
print(df.shape)

(66, 2)


In [121]:
print(df["sentiment"].unique())

['Positive' 'Neutral' 'Negative']


In [122]:
df["sentiment"] = df["sentiment"].str.strip()
df["sentiment"] = df["sentiment"].str.title()

In [123]:
print(encoder.classes_)

['Negative' 'Neutral' 'Positive']


In [124]:
df = pd.read_csv("feedback_dataset.csv")

# Remove extra spaces
df["sentiment"] = df["sentiment"].astype(str).str.strip()

# Standardize labels
df["sentiment"] = df["sentiment"].str.title()

# Remove accidental header rows
df = df[df["sentiment"] != "Sentiment"]

print(df["sentiment"].unique())

['Positive' 'Neutral' 'Negative']


In [125]:
# Remove extra spaces
df["sentiment"] = df["sentiment"].astype(str).str.strip()

# Remove accidental header rows
df = df[df["sentiment"].str.lower() != "sentiment"]

# Keep only valid labels
df = df[df["sentiment"].isin([
    "Positive",
    "Neutral",
    "Negative"
])]

print(df["sentiment"].unique())

['Positive' 'Neutral' 'Negative']


In [126]:
encoder = LabelEncoder()
y = encoder.fit_transform(df["sentiment"])

print(encoder.classes_)

['Negative' 'Neutral' 'Positive']
